### 1. Company Research Summary
Olist is a B2B retail technology company that offers an integrated platform solution for small and medium-sized businesses (SMBs) in Brazil. By connecting sellers to major marketplaces, Olist provides a complete ecosystem that includes e-commerce store management, logistics, payments, and credit services. Their primary goal is to remove the operational hassle for sellers, enabling them to sell more while ensuring a smooth journey for the end consumer. Olist’s core customers are the SMB sellers themselves, and their business model relies on subscription fees and commissions on sales.

### 2. Dataset Understanding & Schema
 The Olist dataset is a relational database consisting of 8 CSV files representing a real-world e-commerce ecosystem. To answer our specific business questions regarding logistics and seller performance, I intentionally selected the 5 core tables that track the order lifecycle:
 - The Core (olist_orders_dataset): Acts as the central hub. It connects a customer_id to an order_id and tracks the lifecycle of the order through timestamps (purchased, approved, shipped, delivered).
 - Items & Sellers (olist_order_items_dataset): Links each order to the specific product_id purchased and the seller_id who sold it, including price and freight value.
 - Feedback (olist_order_reviews_dataset): Contains the 1-to-5 star rating and written comments left by the customer after delivery.
 - Geography (olist_customers_dataset & olist_sellers_dataset): Contains state and zip code information, allowing us to map the physical distance between sellers and buyers.

### 3. Business Hypothesis & Approach
 The Business Question: Where are the operational friction points causing customer dissatisfaction, and how are they impacting Olist's reputation?

 - Step A: The Hypothesis:
 Olist connects small sellers to major marketplaces. When a customer receives a product later than the promised estimated delivery date, they experience frustration. My hypothesis is that late deliveries are the primary driver of 1-star reviews, regardless of the actual product quality.

 - Step B: Mapping to the Data:
 To test this, I will use the order_estimated_delivery_date (what Olist promised) and order_delivered_customer_date (what actually happened) from the Orders table. I will merge this with the review_score from the Reviews table to see if a mathematical correlation exists between delay time and customer satisfaction.



In [10]:
import pandas as pd
orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
customers = pd.read_csv('olist_customers_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')

print(f"orders shape: {orders.shape}")
print("Order Items shape:", order_items.shape)
print("Reviews shape:", reviews.shape)

# Let's look at the columns we have to work with

print("\n orders columns:",orders.columns.tolist())
print("\n Order Items columns:", order_items.columns.tolist())
print("\nsellers columns:", sellers.columns.tolist())
print(reviews.columns.tolist())


orders shape: (99441, 8)
Order Items shape: (112650, 7)
Reviews shape: (99224, 7)

 orders columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

 Order Items columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

sellers columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


### 4. Analysis: The Logistics Friction
To test the hypothesis above, we must calculate the difference between the estimated delivery date and the actual delivery date, then group the results to see the impact on review scores.



In [ ]:
# 1. Convert date columns to actual datetime objects so we can do math on them
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

# 2. calaculting the diff (actual - estimated)
orders['delivery_delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

# 3. Merge the Orders with the Reviews so we can connect "Delay" to "Review Score"
orders_reviews = pd.merge(orders, reviews, on= "order_id", how="inner")

# 4. group the delay into categories 
def delay_cat(days):
    if pd.isna(days):
        return "Unkown"
    elif days <= 0:
        return "early/ ontime"
    elif days  <= 3:
        return "Slightly Late (1-3 days)"
    else :
        return 'Very Late (>3 days)'



orders_reviews["delay_status"] = orders_reviews["delivery_delay_days"].apply(delay_cat)

# 5. summary table: What is the average review score for each delay status?
summary = orders_reviews.groupby("delay_status")["review_score"].mean().sort_values()
print(summary)





delay_status
Unkown                      1.761257
Very Late (>3 days)         1.855951
Slightly Late (1-3 days)    3.289871
early/ ontime               4.289842
Name: review_score, dtype: float64


### 5. Business Interpretation: The Logistics Friction
The Findings:
The data strongly supports the hypothesis. There is a clear, inverse relationship between delivery delays and customer satisfaction:

- Orders delivered On Time or Early receive an average review score of 4.28 out of 5.
- Orders delivered Slightly Late (1-3 days) drop to an average of 3.28.
- Orders that are Very Late (>3 days) plummet to a disastrous average of 1.85 out of 5.

##### What this means for the Business:
This is a massive friction point for Olist. When a customer buys through Olist and the package arrives more than 3 days late, their satisfaction drops to a 1-star rating.
In the marketplace ecosystem, 1-star reviews don't just hurt the individual seller; they hurt Olist's brand reputation. Marketplaces like Amazon penalize or suspend sellers who accumulate 1-star reviews. If Olist cannot control its delivery estimates, sellers will get banned from marketplaces, lose their income, and ultimately churn from the Olist platform.

Olist is currently setting customer expectations (estimated delivery dates) that their logistics network is failing to meet.

### 6.The Second Business Story
In business, the Pareto Principle states that roughly 80% of outcomes come from 20% of causes. For example, 80% of a company's sales often come from just 20% of its customers.

#### The Olist Business Hypothesis
Olist is a B2B company—meaning their actual "customers" are the sellers. Olist makes money by taking a commission on the sales these sellers make.
 
my hypothesis is:
"Olist's revenue is highly concentrated. A tiny fraction of their top sellers are generating the vast majority of their revenue. Meanwhile, the bottom 50% of sellers are bringing in almost no money, yet they still cost Olist money in server space, customer support, and account management."

In [ ]:
# Let's find out how revenue is distributed among Olist's sellers
# 1. Calculate total sales per seller 
seller_sales = order_items.groupby('seller_id')['price'].sum().sort_values(ascending=False)
total_sellers = len(seller_sales)

# 3. Calculate cumulative percentage
seller_cumulative_pct = seller_sales.cumsum() / seller_sales.sum() * 100
# 4. Find out how many sellers make up 50% of total sales
top_50_pct_count = (seller_cumulative_pct <= 50).sum()

print(f"Total active sellers in dataset: {total_sellers}")
print(f"Sellers needed to reach 50% of total sales: {top_50_pct_count}")
print(f"Percentage of sellers driving 50% of sales: {(top_50_pct_count / total_sellers) * 100:.2f}%")



# Look at the bottom 20% of sellers
bottom_20_threshold = seller_sales.quantile(0.20)
print(f"\nSales volume of the bottom 20% of sellers is less than: ${bottom_20_threshold:.2f}")




Total active sellers in dataset: 3095
Sellers needed to reach 50% of total sales: 129
Percentage of sellers driving 50% of sales: 4.17%

Sales volume of the bottom 20% of sellers is less than: $159.76


### 7. Business Interpretation: Seller Concentration Risk

The Findings:
The analysis reveals an extreme concentration of sales volume among Olist's seller base. Out of 3,095 active sellers, only 129 sellers (just 4.17%) are responsible for 50% of Olist's total Gross Merchandise Value (GMV). Conversely, the bottom 20% of sellers have generated less than $159.76 in total sales over the dataset's timeframe.

#### What this means for the Business:
This data exposes two massive risks for Olist:
Revenue Fragility: Olist's revenue is heavily dependent on a tiny fraction of top-performing sellers. If even a small number of these top 129 sellers churn to a competitor or get suspended by marketplaces (due to the late deliveries we identified earlier), Olist's revenue will suffer a severe impact.
Resource Drain: The bottom 20% of sellers are generating negligible sales. However, they still consume Olist's operational resources, including server storage, customer support, and seller onboarding. These "zombie" sellers are likely operating at a loss for the platform.

### 8. Solution Proposal: AI-Assisted Seller Intelligence Suite
Based on the data analysis, HVIA proposes a two-pronged Data & AI solution to protect Olist’s revenue and optimize operational efficiency:

#### Solution 1: Predictive Late-Delivery Alert System (Logistics AI)

The Problem: Deliveries exceeding the estimated date by >3 days cause review scores to plummet to 1.8/5, damaging marketplace reputations.
The HVIA Solution: HVIA will build a Machine Learning classification model that predicts the probability of an order being late at the exact moment of checkout.
Features: Seller historical delay rate, geographical distance to customer, freight value, and seasonal peaks.
Business Impact: If an order is flagged as high-risk for delay, Olist can automatically adjust the estimated delivery date shown to the customer or alert the seller to expedite shipping—preventing 1-star reviews before they happen.

#### Solution 2: Predictive Seller Churn & Tiering Model (Customer Intelligence)

The Problem: 4.17% of sellers drive 50% of revenue. Olist is highly vulnerable if top sellers leave, while the bottom 20% drain operational resources.
The HVIA Solution: HVIA will build a predictive model that monitors seller health metrics (sales trends, review scores, delivery performance).
Business Impact: The model will alert Olist's account managers when a "Top Tier" seller shows signs of churn (e.g., dropping sales), allowing proactive intervention. Additionally, it will flag "Zombie Sellers" in the bottom 20% for automated offboarding or re-engagement campaigns, saving operational costs.